# Chapter 3 · 전력수요 LSTM

**연구질문:** 전력수요 예측에서 LSTM은 트리·선형 모형과 비교해 이점이 있는가?

> 위에서 아래로 실행하면 끝까지 동작합니다. 온라인 실패 시 합성 대체(`DATA_KIND` 표시).

## 1. 핵심 개념·수식

**지도학습 형태의 예측:** $\hat y_{t}=f(y_{t-1},\dots,y_{t-L},\ \text{rolling}_{t-1},\ \text{calendar}_t,\ x_t)$. 모든 특성은 **예측 시점 $t$에 알 수 있는 정보**만 사용한다(롤링 통계는 반드시 한 칸 이동).

**시간순 검증:** 무작위 분할 대신 훈련 구간이 항상 검증 구간보다 앞서는 expanding-window 분할을 쓴다.

**기준선 대비 성능:** $\mathrm{MASE}<1$이면 1단계 나이브보다 낫다. 수익률처럼 예측력이 약한 자료는 **표본 외 $R^2_{OS}=1-\dfrac{\sum(y_t-\hat y_t)^2}{\sum(y_t-\bar y_{t-1})^2}$** ($\bar y_{t-1}$: 직전까지의 평균)로 평가한다.

**변수 중요도:** 순열 중요도는 특성을 섞었을 때 검증 오차가 늘어난 정도이며 인과적 중요도가 아니다. 상관된 특성이 있으면 해석이 불안정하다.

In [ ]:
import os, io, json, subprocess, sys, warnings, urllib.request, urllib.parse
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from scipy import stats, optimize

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)
SEED = 42
# 네트워크 없이 합성자료로만 실행하려면 환경변수 TSLAB_OFFLINE=1 로 실행하거나 아래를 False로 바꾼다.
USE_LIVE_DATA = os.environ.get("TSLAB_OFFLINE") != "1"
plt.rcParams.update({"figure.figsize": (10, 4), "axes.grid": True,
                     "axes.spines.top": False, "axes.spines.right": False})

def ensure(pkg, mod=None):
    # 없으면 설치를 시도한다. 실패해도 노트북은 계속 실행되고 해당 선택 기능만 건너뛴다.
    try:
        __import__(mod or pkg); return True
    except Exception:
        if not USE_LIVE_DATA:
            return False
        try:
            subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=True, timeout=300)
            __import__(mod or pkg); return True
        except Exception as e:
            print(f"[건너뜀] {pkg} 설치/가져오기 실패: {type(e).__name__}")
            return False

def fetch_bytes(url, timeout=60):
    req = urllib.request.Request(url, headers={"User-Agent": "ts-lab-course/1.0 (education)"})
    with urllib.request.urlopen(req, timeout=timeout) as r:
        return r.read()

# ---- 평가 지표와 기준선 ----
def mae(a, f):  return float(np.mean(np.abs(np.asarray(a, float) - np.asarray(f, float))))
def rmse(a, f): return float(np.sqrt(np.mean((np.asarray(a, float) - np.asarray(f, float)) ** 2)))
def mase(a, f, train, m=1):
    train = np.asarray(train, float)
    scale = np.mean(np.abs(train[m:] - train[:-m]))
    return mae(a, f) / scale
def naive_fc(train, h):  return np.repeat(np.asarray(train, float)[-1], h)
def snaive_fc(train, h, m):
    train = np.asarray(train, float)
    return np.tile(train[-m:], int(np.ceil(h / m)))[:h]
def drift_fc(train, h):
    train = np.asarray(train, float)
    slope = (train[-1] - train[0]) / (len(train) - 1)
    return train[-1] + slope * np.arange(1, h + 1)
def show(df, digits=4):
    print(df.round(digits).to_string())

In [ ]:
def parse_openmeteo_daily(obj):
    d = obj["daily"]; idx = pd.to_datetime(d["time"])
    tmax = pd.Series(d["temperature_2m_max"], index=idx, dtype=float)
    tmin = pd.Series(d["temperature_2m_min"], index=idx, dtype=float)
    return ((tmax + tmin) / 2).rename("temp_mid")          # 최고·최저 평균(정의된 일평균 아님)

def parse_openmeteo_pm25(obj):
    h = obj["hourly"]; s = pd.Series(h["pm2_5"], index=pd.to_datetime(h["time"]), dtype=float)
    return s.resample("D").mean().rename("pm2_5")

def parse_daily_min_temp(raw):
    df = pd.read_csv(io.BytesIO(raw)); df.columns = [c.strip().lower() for c in df.columns]
    s = pd.Series(pd.to_numeric(df["temp"], errors="coerce").values, index=pd.to_datetime(df["date"]), name="temp_mid")
    if len(s) < 3000 or not s.dropna().between(-10, 40).all():
        raise ValueError("일 최저기온 길이/범위 이상")
    return s

def synthetic_temp(n=3652, seed=SEED):
    rng = np.random.default_rng(seed); t = np.arange(n)
    return pd.Series(12 + 13 * np.sin(2 * np.pi * (t - 110) / 365.25) + rng.normal(0, 3, n) + 0.0004 * t,
                     index=pd.date_range("2015-01-01", periods=n, freq="D"), name="temp_mid")

def load_energy(which="temp"):
    temp, pm, kind = None, None, "SYNTHETIC (NOT real data)"
    if USE_LIVE_DATA:
        try:
            u = ("https://archive-api.open-meteo.com/v1/archive?latitude=37.57&longitude=126.98"
                 "&start_date=2015-01-01&end_date=2024-12-31&daily=temperature_2m_max,temperature_2m_min&timezone=Asia%2FSeoul")
            temp = parse_openmeteo_daily(json.loads(fetch_bytes(u, timeout=120)))
            kind = "REAL: Open-Meteo/ERA5 재분석 서울 일 기온 (관측소 실측 아님)"
        except Exception as e:
            print("Open-Meteo 기온 수집 실패 -> 2순위 시도:", type(e).__name__, str(e)[:120])
            try:
                temp = parse_daily_min_temp(fetch_bytes("https://raw.githubusercontent.com/jbrownlee/Datasets/master/daily-min-temperatures.csv", timeout=60))
                kind = "REAL: Melbourne 일 최저기온 1981-1990 (jbrownlee/Datasets; 서울 아님, 일평균 아닌 최저기온)"
            except Exception as e2:
                print("일 최저기온 수집 실패 -> 합성 대체:", type(e2).__name__, str(e2)[:100]); temp = None
        try:
            u = ("https://air-quality-api.open-meteo.com/v1/air-quality?latitude=37.57&longitude=126.98"
                 "&hourly=pm2_5&start_date=2022-09-01&end_date=2024-12-31&timezone=Asia%2FSeoul")
            pm = parse_openmeteo_pm25(json.loads(fetch_bytes(u, timeout=120)))
        except Exception as e:
            print("Open-Meteo 대기질 수집 실패(선택 항목):", type(e).__name__, str(e)[:100])
    if temp is None:
        temp = synthetic_temp()
    temp = temp.asfreq("D").interpolate()
    if which == "demand":
        rng = np.random.default_rng(SEED + 5); t = np.arange(len(temp))
        dow = temp.index.dayofweek.to_numpy()
        dem = 60 + 0.9 * np.abs(temp.to_numpy() - 18) ** 1.15 + 6 * (dow < 5) + 0.004 * t + rng.normal(0, 2.0, len(temp))
        return dict(y=pd.Series(dem, index=temp.index, name="demand"), X=temp.to_frame("temp"), m=7, freq="D",
                    label="일별 전력수요(합성, 기온 의존)", unit="GWh",
                    kind="SYNTHETIC-DERIVED (NOT real data): 수요는 합성, 기온 입력은 " + kind)
    return dict(y=temp, pm=pm, m=365, freq="D", label="일평균 기온", unit="°C", kind=kind)

D = load_energy("demand")
y, DATA_KIND, FREQ = D["y"], D["kind"], D["freq"]
M = D["m"]
print("DATA_KIND:", DATA_KIND); print("관측 수:", len(y), "| 기간:", y.index.min().date(), "~", y.index.max().date())

## 2. 특성 구성·모형·평가

In [ ]:
from sklearn.linear_model import Ridge, LogisticRegression
from sklearn.ensemble import (RandomForestRegressor, HistGradientBoostingRegressor,
                              RandomForestClassifier, HistGradientBoostingClassifier)
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import TimeSeriesSplit
from sklearn.inspection import permutation_importance
from sklearn.metrics import roc_auc_score, average_precision_score, precision_recall_curve

def calendar_features(idx, freq):
    out = pd.DataFrame(index=idx)
    if freq in ("D", "B", "h"):
        out["dow"] = idx.dayofweek
    if freq == "h":
        out["hour"] = idx.hour
    out["month_sin"] = np.sin(2 * np.pi * idx.month / 12); out["month_cos"] = np.cos(2 * np.pi * idx.month / 12)
    return out

def lag_features(s, lags, rolls=()):
    X = pd.DataFrame({f"lag_{k}": s.shift(k) for k in lags})
    for w in rolls:
        X[f"rmean_{w}"] = s.shift(1).rolling(w).mean()          # shift(1): 현재 값을 쓰지 않는다(누수 방지)
        X[f"rstd_{w}"] = s.shift(1).rolling(w).std()
    return X

def get_models():
    models = {
        "Ridge": make_pipeline(StandardScaler(), Ridge(alpha=1.0)),
        "RandomForest": RandomForestRegressor(n_estimators=150, min_samples_leaf=5, max_features=0.6, n_jobs=-1, random_state=SEED),
        "HistGB": HistGradientBoostingRegressor(max_iter=250, learning_rate=0.05, random_state=SEED),
    }
    if ensure("xgboost"):                                         # 선택: 설치돼 있으면 XGBoost도 비교
        from xgboost import XGBRegressor
        models["XGBoost"] = XGBRegressor(n_estimators=300, max_depth=4, learning_rate=0.05, subsample=0.8,
                                         random_state=SEED, n_jobs=-1, verbosity=0)
    return models

def fit_eval(X, target, n_test, models=None):
    # 앞쪽으로 학습, 마지막 n_test 관측으로 평가 (1단계 예측, 시차 입력은 실제값)
    data = pd.concat([X, target.rename("target")], axis=1).dropna()
    tr, te = data.iloc[:-n_test], data.iloc[-n_test:]
    Xtr, ytr, Xte, yte = tr.drop(columns="target"), tr["target"], te.drop(columns="target"), te["target"]
    models = models or get_models(); preds = {}
    for name, mdl in models.items():
        mdl.fit(Xtr, ytr); preds[name] = pd.Series(mdl.predict(Xte), index=te.index)
    return dict(Xtr=Xtr, ytr=ytr, Xte=Xte, yte=yte, preds=preds, models=models, data=data)

def cv_rmse(X, target, models, n_splits=4, test_size=None):
    data = pd.concat([X, target.rename("target")], axis=1).dropna()
    test_size = test_size or max(30, len(data) // 10)
    tss = TimeSeriesSplit(n_splits=n_splits, test_size=test_size); out = {k: [] for k in models}
    from sklearn.base import clone
    for tr_i, te_i in tss.split(data):
        a, b = data.iloc[tr_i], data.iloc[te_i]
        for name, mdl in models.items():
            m_ = clone(mdl).fit(a.drop(columns="target"), a["target"])
            out[name].append(rmse(b["target"], m_.predict(b.drop(columns="target"))))
    return pd.DataFrame(out).agg(["mean", "std"]).T.rename(columns={"mean": "CV_RMSE_mean", "std": "CV_RMSE_sd"})

In [ ]:
# 결과표: 기준선과 모델, 마지막 구간 평가 + 시간순 CV
def report(res, baselines, train_for_scale):
    rows = {}
    for name, f in {**baselines, **res["preds"]}.items():
        rows[name] = {"MAE": mae(res["yte"], f), "RMSE": rmse(res["yte"], f),
                      "MASE": mase(res["yte"], f, train_for_scale, m=1)}
    tab = pd.DataFrame(rows).T.sort_values("RMSE"); show(tab); return tab

In [ ]:
s = y.astype(float); temp = D["X"]["temp"]
X = pd.concat([lag_features(s, [1, 2, 7, 14], rolls=(7,)), calendar_features(s.index, "D"), temp.to_frame("temp")], axis=1)
n_test = 365
res = fit_eval(X, s, n_test)
yte = res["yte"]
base = {"naive(lag1)": s.shift(1).loc[yte.index], "seasonal naive(lag7)": s.shift(7).loc[yte.index]}
tab = report(res, base, res["ytr"].to_numpy())

## 3. LSTM

In [ ]:
# LSTM: 최근 W일의 [수요, 기온]을 입력으로 다음 날 수요를 예측. torch가 없으면 MLP로 대체(표시).
W = 14
HAS_TORCH = ensure("torch")
arr = np.column_stack([s.to_numpy(float), temp.to_numpy(float)])
n_tr = len(arr) - n_test
mu_, sd_ = arr[:n_tr].mean(axis=0), arr[:n_tr].std(axis=0)          # 스케일은 학습구간 통계만 사용(누수 방지)
Z = (arr - mu_) / sd_
Xw = np.stack([Z[i - W:i] for i in range(W, len(Z))]); yw = Z[W:, 0]
cut = n_tr - W                                                       # 학습 윈도우의 마지막 인덱스
if HAS_TORCH:
    import torch, torch.nn as nn
    torch.manual_seed(SEED)
    class LSTMReg(nn.Module):
        def __init__(self, n_in, hid=32):
            super().__init__(); self.lstm = nn.LSTM(n_in, hid, batch_first=True); self.fc = nn.Linear(hid, 1)
        def forward(self, x):
            o, _ = self.lstm(x); return self.fc(o[:, -1, :]).squeeze(-1)
    net = LSTMReg(Xw.shape[2]); opt = torch.optim.Adam(net.parameters(), lr=1e-2); lossf = nn.MSELoss()
    Xt = torch.tensor(Xw[:cut], dtype=torch.float32); yt = torch.tensor(yw[:cut], dtype=torch.float32)
    for epoch in range(25):
        net.train(); perm = torch.randperm(len(Xt))
        for i in range(0, len(Xt), 64):
            b = perm[i:i + 64]; opt.zero_grad(); loss = lossf(net(Xt[b]), yt[b]); loss.backward(); opt.step()
    net.eval()
    with torch.no_grad():
        p_te = net(torch.tensor(Xw[cut:], dtype=torch.float32)).numpy()
    kind_nn = "LSTM(torch)"
else:
    from sklearn.neural_network import MLPRegressor
    mlp = MLPRegressor(hidden_layer_sizes=(32,), max_iter=300, random_state=SEED).fit(Xw[:cut].reshape(cut, -1), yw[:cut])
    p_te = mlp.predict(Xw[cut:].reshape(len(Xw) - cut, -1)); kind_nn = "MLP(torch 없음: LSTM 대체)"
p_te = p_te * sd_[0] + mu_[0]                                        # 원척도 복원
y_true = s.to_numpy()[n_tr:]
print(f"{kind_nn}: RMSE={rmse(y_true, p_te):.3f}  MAE={mae(y_true, p_te):.3f}")
tab.loc[kind_nn] = {"MAE": mae(y_true, p_te), "RMSE": rmse(y_true, p_te), "MASE": mase(y_true, p_te, res["ytr"].to_numpy(), m=1)}
show(tab.sort_values("RMSE"))

**읽는 법:** 수요는 기온의 비선형(U자) 함수로 합성했다. 선형모형이 놓치는 비선형을 트리·신경망이 잡는지가 비교 포인트이며, 자료가 작고 구조가 단순해 신경망의 이점이 작게 나올 수 있다. 신경망은 초기값·epoch에 민감하므로 여러 시드로 반복한 뒤 결론을 내려야 한다.

## 3. 결과 해석 작성
1. 기준선(나이브·계절 나이브·역사평균)보다 나은 모델이 있는가? 개선폭은 CV 변동성($sd$) 대비 큰가?
2. 변수 중요도가 도메인 지식과 일관되는가? (인과적 해석 금지)
3. 가장 위험한 누수 경로 3개를 적고, 이 노트북에서 어떻게 막았는지(롤링 shift, 학습구간 통계 스케일, 시간순 분할)를 서술한다.
4. `DATA_KIND`가 합성이면 결과는 **방법 점검**이다. 실제 자료로 바꿨을 때 달라질 점을 적는다.

**제출:** 결과표(기준선 포함), CV 표, 중요도 그래프/표, 해석 단락, 한계.